In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


In [2]:
!pip install -q transformers accelerate bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 29.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 83.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cudf-cu12 2

In [3]:
import numpy as np
import pandas as pd
import torch

train_df = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")
test_df  = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv")
OPTION_COLS = ["A", "B", "C", "D", "E"]

def mapk(actual, predicted, k=3):
    def apk(a, p):
        score, hits = 0.0, 0
        for i, pi in enumerate(p[:k]):
            if pi == a:
                hits += 1
                score += hits / (i + 1)
        return score
    return np.mean([apk(a, p) for a, p in zip(actual, predicted)])

print(train_df.shape, test_df.shape)

(2000, 8) (500, 7)


In [4]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    dtype=torch.float16,
    device_map="auto"
)
model.eval()
print("Qwen2.5 loaded!")

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Qwen2.5 loaded!


In [5]:
def predict_top3(prompt, options):
    opts_text = "\n".join([f"{OPTION_COLS[i]}) {options[i][:300]}" 
                           for i in range(5)])
    
    messages = [
        {"role": "system", "content": "You are an expert scientist answering multiple choice questions. Focus on subtle differences between options."},
        {"role": "user", "content": f"""Question: {prompt}

Options:
{opts_text}

Rank the top 3 correct answers from most likely to least likely.
Reply with ONLY 3 letters separated by spaces. Example: A C B"""}
    ]
    
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(text, return_tensors="pt").to(DEVICE)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=15,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )
    
    new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    generated = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
    print(f"Raw output: {generated}")  # debug — remove later
    
    # Parse letters
    found = []
    for ch in generated.upper():
        if ch in OPTION_COLS and ch not in found:
            found.append(ch)
        if len(found) == 3:
            break
    
    # Fill missing slots
    for col in OPTION_COLS:
        if col not in found:
            found.append(col)
        if len(found) == 3:
            break
    
    return found[:3]

# Quick test on 1 row
row = train_df.iloc[0]
opts = [row[c] for c in OPTION_COLS]
print("Prediction:", predict_top3(row["prompt"], opts))
print("Actual:", row["answer"])

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Raw output: B A C
Prediction: ['B', 'A', 'C']
Actual: B


In [6]:
from sklearn.model_selection import train_test_split

tr, val = train_test_split(train_df, test_size=0.2, random_state=42)
val = val.reset_index(drop=True)

print(f"Validating on {len(val)} samples...")

preds, trues = [], []
for i, row in val.iterrows():
    opts = [row[c] for c in OPTION_COLS]
    pred = predict_top3(row["prompt"], opts)
    preds.append(pred)
    trues.append(row["answer"])
    if (i+1) % 50 == 0:
        print(f"  {i+1}/{len(val)} | MAP@3: {mapk(trues, preds):.4f}")

print(f"\nFinal Validation MAP@3: {mapk(trues, preds):.4f}")

Validating on 400 samples...
Raw output: D C B
Raw output: E C A
Raw output: A E B
Raw output: B C E
Raw output: C B A
Raw output: E D C
Raw output: C B E
Raw output: A B E
Raw output: A E D
Raw output: B C D
Raw output: C A D
Raw output: B C A
Raw output: A C B
Raw output: D A E
Raw output: B C D
Raw output: B C E
Raw output: B C A
Raw output: B A C
Raw output: C A B
Raw output: C D E
Raw output: C A D
Raw output: E B C
Raw output: D A B
Raw output: C A B
Raw output: B C A
Raw output: B C A
Raw output: C A B
Raw output: C A B
Raw output: E C B
Raw output: B C A
Raw output: A B E
Raw output: B C A
Raw output: D C E
Raw output: B C A
Raw output: B C D
Raw output: E B A
Raw output: D A E
Raw output: B A C
Raw output: C D B
Raw output: C D E
Raw output: A C B
Raw output: B C A
Raw output: B C A
Raw output: E D C
Raw output: B C A
Raw output: A C B
Raw output: D C B
Raw output: A B E
Raw output: C A D
Raw output: D A E
  50/400 | MAP@3: 0.8900
Raw output: C B E
Raw output: C B A
Raw output

In [7]:
test_preds = []

for i, row in test_df.iterrows():
    opts = [row[c] for c in OPTION_COLS]
    pred = predict_top3(row["prompt"], opts)
    test_preds.append(pred)
    if (i+1) % 100 == 0:
        print(f"{i+1}/{len(test_df)} done")

submission = pd.DataFrame({
    "id":         test_df["id"],
    "Prediction": [" ".join(p) for p in test_preds]
})
submission.to_csv("submission.csv", index=False)
print("Done!")
print(submission.head(10))

Raw output: A E D
Raw output: B A C
Raw output: B C E
Raw output: E C D
Raw output: C A B
Raw output: B C A
Raw output: E D C
Raw output: B C A
Raw output: A C D
Raw output: B B B
Raw output: A B E
Raw output: D C B
Raw output: C A B
Raw output: B C D
Raw output: E D C
Raw output: B C D
Raw output: D E C
Raw output: B C E
Raw output: A B E
Raw output: D C B
Raw output: A C B
Raw output: C D E
Raw output: D C E
Raw output: E C A
Raw output: D C B
Raw output: E C A
Raw output: E D C
Raw output: C B A
Raw output: B D E
Raw output: A C B
Raw output: E C D
Raw output: C B A
Raw output: D C E
Raw output: C B E
Raw output: B C D
Raw output: C A B
Raw output: C B E
Raw output: C A B
Raw output: D B E
Raw output: C E A
Raw output: A B E
Raw output: E C B
Raw output: B D E
Raw output: A B E
Raw output: D A E
Raw output: B A C
Raw output: D A E
Raw output: A B C
Raw output: B A C
Raw output: C B E
Raw output: D C E
Raw output: A C E
Raw output: C A B
Raw output: E C A
Raw output: B C A
Raw output

In [8]:
print(f"Validation MAP@3: {mapk(trues, preds):.4f}")

Validation MAP@3: 0.8600


In [9]:
# OPTION_COLS = ["A", "B", "C", "D", "E"]

# def convert_mmlu(dataset):
#     rows = []
#     for item in dataset:
#         choices = item["choices"]
#         # MMLU has 4 choices, pad to 5
#         while len(choices) < 5:
#             choices.append("None of the above")
#         answer_idx = item["answer"]  # 0-3
#         answer_letter = OPTION_COLS[answer_idx]
#         row = {
#             "prompt": item["question"],
#             "A": choices[0],
#             "B": choices[1],
#             "C": choices[2],
#             "D": choices[3],
#             "E": choices[4],
#             "answer": answer_letter
#         }
#         rows.append(row)
#     return pd.DataFrame(rows)

# def convert_arc(dataset):
#     rows = []
#     for item in dataset:
#         choices = item["choices"]["text"]
#         labels  = item["choices"]["label"]
#         # Map to A-E
#         while len(choices) < 5:
#             choices.append("None of the above")
#             labels.append(OPTION_COLS[len(labels)])
#         answer_letter = item["answerKey"]
#         if answer_letter not in OPTION_COLS:
#             continue
#         row = {
#             "prompt": item["question"],
#             "A": choices[0], "B": choices[1],
#             "C": choices[2], "D": choices[3],
#             "E": choices[4],
#             "answer": answer_letter
#         }
#         rows.append(row)
#     return pd.DataFrame(rows)

# mmlu_df = convert_mmlu(mmlu)
# arc_df  = convert_arc(arc)

# # Your original data
# train_df = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")
# test_df  = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv")

# # Combine all — your data gets 3x weight since it's most similar
# combined_df = pd.concat([
#     train_df,
#     train_df,       # repeat your data 2 more times
#     train_df,
#     mmlu_df,
#     arc_df
# ], ignore_index=True).sample(frac=1, random_state=42).reset_index(drop=True)

# print(f"Combined training data: {len(combined_df)} rows")
# print(f"Your data: {len(train_df)} | MMLU: {len(mmlu_df)} | ARC: {len(arc_df)}")

In [10]:
# # In CELL 4, change this line:
# # tr_df, val_df = train_test_split(train_df, ...)
# # TO:
# from sklearn.model_selection import train_test_split

# # Val ONLY from your original data — not MMLU/ARC
# val_df = train_df.sample(frac=0.15, random_state=42)
# tr_df  = combined_df  # train on everything

# tr_df  = tr_df.reset_index(drop=True)
# val_df = val_df.reset_index(drop=True)

# print(f"Train: {len(tr_df)} | Val: {len(val_df)}")

In [11]:
# def predict_top3(prompt, options):
#     scores = []
    
#     for opt in options:
#         # Truncate long options but keep enough for reasoning
#         opt_text = opt[:400]
        
#         # Encode question + option as NLI pair
#         enc = tokenizer(
#             prompt,
#             opt_text,
#             return_tensors="pt",
#             truncation=True,
#             max_length=512,
#             padding=True
#         ).to(DEVICE)
        
#         with torch.no_grad():
#             logits = model(**enc).logits
        
#         probs = F.softmax(logits, dim=-1)[0]
        
#         # Get entailment score (label depends on model)
#         # Print labels in cell 3 to confirm index
#         entail_score = probs[-1].item()  # usually last = entailment
#         scores.append(entail_score)
    
#     scores = np.array(scores)
#     top3_idx = np.argsort(scores)[::-1][:3]
#     return [OPTION_COLS[i] for i in top3_idx]

# # Test on first row
# row = train_df.iloc[0]
# opts = [row[c] for c in OPTION_COLS]
# result = predict_top3(row["prompt"], opts)
# print("Prediction:", result)
# print("Actual:", row["answer"])

In [12]:
# DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
# print("Device:", DEVICE)

# # This model is specifically trained on NLI + MCQ tasks
# # Much stronger than generic DeBERTa for answer ranking
# MODEL_NAME = "SEBIS/deberta-v3-large-zeroshot-v2"

# tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
# model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME)
# model = model.to(DEVICE)
# model.eval()
# print("Model loaded!")
# print("Labels:", model.config.id2label)

In [13]:
# from sklearn.model_selection import train_test_split

# tr_df, val_df = train_test_split(train_df, test_size=0.2, random_state=42)
# val_df = val_df.reset_index(drop=True)
# val_sample = val_df.head(200).reset_index(drop=True)

# preds, trues = [], []
# for i, row in val_sample.iterrows():
#     opts = [row[c] for c in OPTION_COLS]
#     pred = predict_top3(row["prompt"], opts)
#     preds.append(pred)
#     trues.append(row["answer"])
#     if (i+1) % 50 == 0:
#         print(f"  {i+1}/200 | MAP@3: {mapk(trues, preds):.4f}")

# print(f"\nValidation MAP@3: {mapk(trues, preds):.4f}")

In [14]:
# test_preds = []

# for i, row in test_df.iterrows():
#     opts = [row[c] for c in OPTION_COLS]
#     pred = predict_top3(row["prompt"], opts)
#     test_preds.append(pred)
#     if (i+1) % 100 == 0:
#         print(f"{i+1}/{len(test_df)} done")

# submission = pd.DataFrame({
#     "id":         test_df["id"],
#     "Prediction": [" ".join(p) for p in test_preds]
# })
# submission.to_csv("submission.csv", index=False)
# print("Done!")
# print(submission.head(10))

In [15]:
# def predict_top3(prompt, options):
#     # Encode prompt
#     prompt_emb = model.encode(prompt, normalize_embeddings=True, 
#                               convert_to_tensor=True)
    
#     # Encode all 5 options
#     option_embs = model.encode(
#         [opt[:400] for opt in options],
#         normalize_embeddings=True,
#         convert_to_tensor=True
#     )
    
#     # Cosine similarity between prompt and each option
#     scores = util.cos_sim(prompt_emb, option_embs)[0].cpu().numpy()
    
#     top3_idx = np.argsort(scores)[::-1][:3]
#     return [OPTION_COLS[i] for i in top3_idx]

# # Quick test
# row = train_df.iloc[0]
# opts = [row[c] for c in OPTION_COLS]
# print("Prediction:", predict_top3(row["prompt"], opts))
# print("Actual:", row["answer"])

In [16]:
# from sklearn.model_selection import train_test_split

# tr_df, val_df = train_test_split(train_df, test_size=0.2, random_state=42)
# val_df = val_df.reset_index(drop=True)

# preds, trues = [], []
# for i, row in val_df.iterrows():
#     opts = [row[c] for c in OPTION_COLS]
#     pred = predict_top3(row["prompt"], opts)
#     preds.append(pred)
#     trues.append(row["answer"])
#     if (i+1) % 50 == 0:
#         print(f"  {i+1}/{len(val_df)} | MAP@3: {mapk(trues, preds):.4f}")

# print(f"\nValidation MAP@3: {mapk(trues, preds):.4f}")

In [17]:
# test_preds = []

# for i, row in test_df.iterrows():
#     opts = [row[c] for c in OPTION_COLS]
#     pred = predict_top3(row["prompt"], opts)
#     test_preds.append(pred)
#     if (i+1) % 100 == 0:
#         print(f"{i+1}/{len(test_df)} done")

# submission = pd.DataFrame({
#     "id":         test_df["id"],
#     "Prediction": [" ".join(p) for p in test_preds]
# })
# submission.to_csv("submission.csv", index=False)
# print("Done!")
# print(submission.head(10))

In [18]:
# data_collator = DataCollatorForMultipleChoice(tokenizer)

# args = TrainingArguments(
#     output_dir="./deberta-large-mcq",
#     per_device_train_batch_size=2,
#     per_device_eval_batch_size=4,
#     gradient_accumulation_steps=8,
#     num_train_epochs=8,
#     learning_rate=1e-4,
#     warmup_steps=200,
#     weight_decay=0.01,
#     lr_scheduler_type="cosine",
#     bf16=True,
#     fp16=False,
#     eval_strategy="epoch",
#     save_strategy="epoch",
#     load_best_model_at_end=True,
#     metric_for_best_model="map3",
#     greater_is_better=True,
#     logging_steps=10,
#     report_to="none",
#     label_names=["labels"],
# )

# trainer = Trainer(
#     model=model,
#     args=args,
#     train_dataset=tr_dataset,
#     eval_dataset=val_dataset,
#     processing_class=tokenizer,
#     data_collator=data_collator,
#     compute_metrics=compute_metrics,
# )

# trainer.train()
# print("Training done!")

In [19]:
# preds_out = trainer.predict(val_dataset)
# logits = preds_out.predictions
# labels = preds_out.label_ids

# top3_preds = np.argsort(-logits, axis=1)[:, :3]
# pred_labels = [[OPTION_COLS[i] for i in row] for row in top3_preds]
# true_labels = [OPTION_COLS[l] for l in labels]

# print(f"Validation MAP@3: {mapk(true_labels, pred_labels):.4f}")

In [20]:
# # Add dummy labels to test dataset so collator doesn't crash
# test_dataset_with_dummy = test_dataset.map(lambda x: {"labels": 0})

# test_out = trainer.predict(test_dataset_with_dummy)
# test_logits = test_out.predictions
# test_top3 = np.argsort(-test_logits, axis=1)[:, :3]
# test_pred_labels = [[OPTION_COLS[i] for i in row] for row in test_top3]

# submission = pd.DataFrame({
#     "id":         test_df["id"],
#     "Prediction": [" ".join(p) for p in test_pred_labels]
# })
# submission.to_csv("submission.csv", index=False)
# print("Done!")
# print(submission.head(10))

In [21]:
# def predict_top3(prompt, options):
#     # Show FULL options, not truncated — your options are long paragraphs
#     # and the answer is often in the details
#     opts_text = "\n\n".join([
#         f"Option {OPTION_COLS[i]}:\n{options[i]}" 
#         for i in range(5)
#     ])
    
#     messages = [
#         {
#             "role": "system", 
#             "content": (
#                 "You are a science expert. You will be given a multiple choice question "
#                 "with 5 options. The options may look very similar with only subtle differences. "
#                 "Read every word carefully. Identify which option is most accurate and complete. "
#                 "You must respond with exactly 3 letters separated by spaces, ranked best to worst. "
#                 "Only use letters A, B, C, D, or E. Never repeat a letter."
#             )
#         },
#         {
#             "role": "user", 
#             "content": (
#                 f"Question: {prompt}\n\n"
#                 f"{opts_text}\n\n"
#                 "Which 3 options are most likely correct, ranked from best to worst?\n"
#                 "Answer with exactly 3 letters separated by spaces (example: B A C):"
#             )
#         }
#     ]
    
#     text = tokenizer.apply_chat_template(
#         messages, tokenize=False, add_generation_prompt=True
#     )
#     inputs = tokenizer(text, return_tensors="pt", 
#                        truncation=True, max_length=2048).to(DEVICE)
    
#     with torch.no_grad():
#         outputs = model.generate(
#             **inputs,
#             max_new_tokens=20,
#             do_sample=False,
#             repetition_penalty=1.1,
#             pad_token_id=tokenizer.eos_token_id
#         )
    
#     new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
#     generated = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
    
#     # Robust parsing — handle "B A C", "B, A, C", "BAC", etc.
#     import re
#     # Find all valid letters in order
#     found = []
#     # First try to find space/comma separated pattern
#     matches = re.findall(r'\b([ABCDE])\b', generated.upper())
#     for m in matches:
#         if m not in found:
#             found.append(m)
#         if len(found) == 3:
#             break
    
#     # If still not enough, scan char by char
#     if len(found) < 3:
#         for ch in generated.upper():
#             if ch in OPTION_COLS and ch not in found:
#                 found.append(ch)
#             if len(found) == 3:
#                 break
    
#     # Fill remaining with unused options
#     for col in OPTION_COLS:
#         if col not in found:
#             found.append(col)
#         if len(found) == 3:
#             break
    
#     return found[:3]

# # Quick test
# row = train_df.iloc[0]
# opts = [row[c] for c in OPTION_COLS]
# result = predict_top3(row["prompt"], opts)
# print("Prediction:", result)
# print("Actual:", row["answer"])

In [22]:
# def find_unique_part(option, all_options):
#     """Extract the part of this option that differs from others."""
#     option_words = set(option.lower().split())
#     other_words = set()
#     for other in all_options:
#         if other != option:
#             other_words.update(other.lower().split())
#     # Words unique to this option
#     unique_words = option_words - other_words
#     # Find sentences containing unique words
#     sentences = option.split('. ')
#     unique_sentences = [s for s in sentences 
#                        if any(w in s.lower() for w in unique_words)]
#     if unique_sentences:
#         return '. '.join(unique_sentences[:2])
#     return option[:200]  # fallback

# def score_option(prompt, option_text, all_options):
#     # Score both full option AND the unique distinguishing part
#     unique_part = find_unique_part(option_text, all_options)
    
#     # Use the unique part for scoring — this is what matters
#     text_to_score = f"{option_text[:200]} [...] {unique_part}"
    
#     input_text = (
#         f"The following is a science question with its correct answer.\n\n"
#         f"Question: {prompt}\n\n"
#         f"Answer: {text_to_score}"
#     )
    
#     inputs = tokenizer(
#         input_text,
#         return_tensors="pt",
#         truncation=True,
#         max_length=512
#     ).to(DEVICE)
    
#     question_text = (
#         f"The following is a science question with its correct answer.\n\n"
#         f"Question: {prompt}\n\n"
#         f"Answer: "
#     )
#     question_ids = tokenizer(
#         question_text,
#         return_tensors="pt",
#         truncation=True,
#         max_length=512
#     ).input_ids.to(DEVICE)
    
#     answer_start = question_ids.shape[1]
    
#     with torch.no_grad():
#         outputs = model(**inputs, labels=inputs["input_ids"])
#         logits = outputs.logits
#         shift_logits = logits[0, answer_start-1:-1, :]
#         shift_labels = inputs["input_ids"][0, answer_start:]
        
#         if shift_labels.shape[0] == 0:
#             return 0.0
        
#         log_probs = torch.nn.functional.log_softmax(shift_logits, dim=-1)
#         token_log_probs = log_probs[
#             torch.arange(shift_labels.shape[0]), shift_labels
#         ]
#         score = token_log_probs.mean().item()
    
#     return score

# def predict_top3(prompt, options):
#     scores = [score_option(prompt, opt, options) for opt in options]
#     scores = np.array(scores)
#     top3_idx = np.argsort(scores)[::-1][:3]
#     return [OPTION_COLS[i] for i in top3_idx]

In [23]:
# from sklearn.model_selection import train_test_split

# tr, val = train_test_split(train_df, test_size=0.2, random_state=42)
# val = val.reset_index(drop=True)

# print(f"Validating on {len(val)} samples...")

# preds, trues = [], []
# for i, row in val.iterrows():
#     opts = [row[c] for c in OPTION_COLS]
#     pred = predict_top3(row["prompt"], opts)
#     preds.append(pred)
#     trues.append(row["answer"])
#     if (i+1) % 50 == 0:
#         print(f"  {i+1}/{len(val)} | MAP@3: {mapk(trues, preds):.4f}")

# print(f"\nFinal Validation MAP@3: {mapk(trues, preds):.4f}")

In [24]:
# test_preds = []

# for i, row in test_df.iterrows():
#     opts = [row[c] for c in OPTION_COLS]
#     pred = predict_top3(row["prompt"], opts)
#     test_preds.append(pred)
#     if (i+1) % 100 == 0:
#         print(f"{i+1}/{len(test_df)} done")

# submission = pd.DataFrame({
#     "id":         test_df["id"],
#     "Prediction": [" ".join(p) for p in test_preds]
# })
# submission.to_csv("submission.csv", index=False)
# print("Done!")
# print(submission.head(10))

In [25]:
# print(f"Validation MAP@3: {mapk(trues, preds):.4f}")

In [26]:
# ##MILESTONE - 1

# import pandas as pd
# import numpy as np
# import string
# from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS
# from sklearn.metrics.pairwise import cosine_similarity

# # Load the data
# train_df = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv')
# test_df = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv')

# # ============================================
# # Question 1: Frequency distribution of correct answers
# # ============================================
# print("=" * 50)
# print("QUESTION 1: Frequency Distribution of Correct Answers")
# print("=" * 50)

# answer_freq = train_df['answer'].value_counts().sort_index()
# print("Frequency of each answer:")
# print(answer_freq)

# most_frequent = answer_freq.max()
# least_frequent = answer_freq.min()
# sum_most_least = most_frequent + least_frequent

# print(f"\nMost frequent option count: {most_frequent}")
# print(f"Least frequent option count: {least_frequent}")
# print(f"Sum of most and least frequent: {sum_most_least}")

# # ============================================
# # Question 2: Vocabulary size after cleaning prompts
# # ============================================
# print("\n" + "=" * 50)
# print("QUESTION 2: Vocabulary Size After Cleaning")
# print("=" * 50)

# def clean_text(text):
#     # Convert to lowercase
#     text = text.lower()
#     # Remove punctuation
#     text = text.translate(str.maketrans('', '', string.punctuation))
#     return text

# # Clean all prompts
# train_df['cleaned_prompt'] = train_df['prompt'].apply(clean_text)

# # Get all unique words
# all_words = set()
# for prompt in train_df['cleaned_prompt']:
#     words = prompt.split()
#     all_words.update(words)

# vocab_size = len(all_words)
# print(f"Total unique words (vocabulary size): {vocab_size}")

# # ============================================
# # Question 3: Words left in Row ID 1 after removing stop words
# # ============================================
# print("\n" + "=" * 50)
# print("QUESTION 3: Words in Row ID 1 After Removing Stop Words")
# print("=" * 50)

# # Get cleaned prompt for Row ID 1
# row1_prompt = train_df[train_df['id'] == 1]['cleaned_prompt'].values[0]

# # Split into words
# row1_words = row1_prompt.split()

# # Filter out stop words
# row1_filtered = [word for word in row1_words if word not in ENGLISH_STOP_WORDS]

# words_left = len(row1_filtered)
# print(f"Row ID 1 cleaned prompt (first 100 chars): {row1_prompt[:100]}...")
# print(f"Words left after removing stop words: {words_left}")

# # ============================================
# # Question 4: TF-IDF Vectorizer vocabulary size
# # ============================================
# print("\n" + "=" * 50)
# print("QUESTION 4: TF-IDF Vectorizer Vocabulary Size")
# print("=" * 50)

# # Combine prompt and all options into single documents for each row
# combined_texts = []
# for idx, row in train_df.iterrows():
#     # Combine prompt with all options
#     combined = row['prompt'] + ' ' + row['A'] + ' ' + row['B'] + ' ' + row['C'] + ' ' + row['D'] + ' ' + row['E']
#     combined_texts.append(combined)

# # Fit TF-IDF vectorizer
# tfidf_vectorizer = TfidfVectorizer(stop_words='english')
# tfidf_vectorizer.fit(combined_texts)

# feature_columns = len(tfidf_vectorizer.get_feature_names_out())
# print(f"Number of feature columns (vocabulary size): {feature_columns}")

# # ============================================
# # Question 5: Cosine similarity between prompt and option A for Row ID 1
# # ============================================
# print("\n" + "=" * 50)
# print("QUESTION 5: Cosine Similarity for Row ID 1 (Prompt vs Option A)")
# print("=" * 50)

# # Get Row ID 1 data
# row1 = train_df[train_df['id'] == 1].iloc[0]

# # Transform prompt and option A separately
# prompt_vector = tfidf_vectorizer.transform([row1['prompt']])
# option_a_vector = tfidf_vectorizer.transform([row1['A']])

# # Calculate cosine similarity
# similarity_score = cosine_similarity(prompt_vector, option_a_vector)[0][0]

# print(f"Prompt: {row1['prompt'][:100]}...")
# print(f"Option A: {row1['A'][:100]}...")
# print(f"Cosine similarity score: {similarity_score:.4f}")

# # ============================================
# # Question 6: Percentage where highest similarity matches correct answer
# # ============================================
# print("\n" + "=" * 50)
# print("QUESTION 6: Percentage of Highest Similarity Matching Correct Answer")
# print("=" * 50)

# correct_matches = 0
# total_rows = len(train_df)

# for idx, row in train_df.iterrows():
#     # Vectorize prompt
#     prompt_vec = tfidf_vectorizer.transform([row['prompt']])
    
#     # Vectorize each option and calculate similarity
    
#     similarities = {}
#     for option in ['A', 'B', 'C', 'D', 'E']:
#         option_vec = tfidf_vectorizer.transform([row[option]])
#         sim = cosine_similarity(prompt_vec, option_vec)[0][0]
#         similarities[option] = sim
    
#     # Find option with highest similarity
#     highest_sim_option = max(similarities, key=similarities.get)
    
#     # Check if matches correct answer
#     if highest_sim_option == row['answer']:
#         correct_matches += 1

# percentage = (correct_matches / total_rows) * 100
# print(f"Rows where highest similarity matches correct answer: {correct_matches}/{total_rows}")
# print(f"Percentage: {percentage:.2f}%")

# # ============================================
# # Question 7: MAP@3 score for prediction C A B when answer is C
# # ============================================
# print("\n" + "=" * 50)
# print("QUESTION 7: MAP@3 for prediction C A B (answer is C)")
# print("=" * 50)

# def calculate_map_at_3(ground_truth, predictions):
#     """
#     Calculate MAP@3 for a single question
#     predictions: list of 3 predicted answers in order
#     """
#     for i, pred in enumerate(predictions):
#         if pred == ground_truth:
#             return 1.0 / (i + 1)  # 1/k where k is the position (1-indexed)
#     return 0.0  # Not in top 3

# # Example: answer is C, prediction is C A B
# map_score_q7 = calculate_map_at_3('C', ['C', 'A', 'B'])
# print(f"Ground truth: C, Prediction: C A B")
# print(f"MAP@3 score: {map_score_q7}")

# # ============================================
# # Question 8: MAP@3 score for prediction D B E when answer is B
# # ============================================
# print("\n" + "=" * 50)
# print("QUESTION 8: MAP@3 for prediction D B E (answer is B)")
# print("=" * 50)

# map_score_q8 = calculate_map_at_3('B', ['D', 'B', 'E'])
# print(f"Ground truth: B, Prediction: D B E")
# print(f"MAP@3 score: {map_score_q8}")

# # ============================================
# # Question 9: Majority Class Baseline MAP@3
# # ============================================
# print("\n" + "=" * 50)
# print("QUESTION 9: Majority Class Baseline MAP@3")
# print("=" * 50)

# # Get frequency of answers
# answer_counts = train_df['answer'].value_counts()
# print("Answer frequencies:")
# print(answer_counts)

# # Get top 3 most frequent answers
# top3_answers = answer_counts.head(3).index.tolist()
# print(f"Top 3 most frequent answers: {top3_answers}")

# # Calculate MAP@3 for majority baseline
# majority_scores = []
# for idx, row in train_df.iterrows():
#     ground_truth = row['answer']
#     predictions = top3_answers  # Always predict the same top 3
#     score = calculate_map_at_3(ground_truth, predictions)
#     majority_scores.append(score)

# overall_majority_map = np.mean(majority_scores)
# print(f"Overall MAP@3 for Majority Class Baseline: {overall_majority_map:.4f}")

# # ============================================
# # Question 10: TF-IDF Pipeline MAP@3
# # ============================================
# print("\n" + "=" * 50)
# print("QUESTION 10: TF-IDF Pipeline MAP@3")
# print("=" * 50)

# tfidf_scores = []

# for idx, row in train_df.iterrows():
#     # Vectorize prompt
#     prompt_vec = tfidf_vectorizer.transform([row['prompt']])
    
#     # Calculate similarity for each option
#     similarities = {}
#     for option in ['A', 'B', 'C', 'D', 'E']:
#         option_vec = tfidf_vectorizer.transform([row[option]])
#         sim = cosine_similarity(prompt_vec, option_vec)[0][0]
#         similarities[option] = sim
    
#     # Sort options by similarity (highest to lowest)
#     sorted_options = sorted(similarities.items(), key=lambda x: x[1], reverse=True)
    
#     # Get top 3 predictions
#     top3_predictions = [option for option, sim in sorted_options[:3]]
    
#     # Calculate MAP@3 for this row
#     ground_truth = row['answer']
#     score = calculate_map_at_3(ground_truth, top3_predictions)
#     tfidf_scores.append(score)

# overall_tfidf_map = np.mean(tfidf_scores)
# print(f"Overall MAP@3 for TF-IDF Pipeline: {overall_tfidf_map:.4f}")

# # ============================================
# # Create sample submission file
# # ============================================
# print("\n" + "=" * 50)
# print("Creating Sample Submission File")
# print("=" * 50)

# # Create predictions for test set using TF-IDF approach
# submission_predictions = []

# for idx, row in test_df.iterrows():
#     prompt_vec = tfidf_vectorizer.transform([row['prompt']])
    
#     similarities = {}
#     for option in ['A', 'B', 'C', 'D', 'E']:
#         option_vec = tfidf_vectorizer.transform([row[option]])
#         sim = cosine_similarity(prompt_vec, option_vec)[0][0]
#         similarities[option] = sim
    
#     sorted_options = sorted(similarities.items(), key=lambda x: x[1], reverse=True)
#     top3_predictions = [option for option, sim in sorted_options[:3]]
    
#     submission_predictions.append({
#         'ID': row['id'],
#         'Prediction': ' '.join(top3_predictions)
#     })

# # Create submission DataFrame
# submission_df = pd.DataFrame(submission_predictions)

# # Save to CSV
# submission_df.to_csv('sample_submission.csv', index=False)

# print(f"Sample submission file created with {len(submission_df)} predictions")
# print("\nFirst few predictions:")
# print(submission_df.head())